In [ ]:
# ============================================
# 05_YOLOv8.ipynb
# YOLOv8 CLASSIFICATION - DETEKSI KANTUK
# ============================================

import os
import sys
import json
import time
import datetime
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import shutil

# ============================================
# SETUP
# ============================================
warnings.filterwarnings('ignore')
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

print("="*60)
print("🎯 YOLOv8 CLASSIFICATION - DETEKSI KANTUK")
print("="*60)

# ============================================
# 1. SYSTEM INFO
# ============================================

print(f"\n[System Info]")

# Cek ultralytics
try:
    import ultralytics
    print(f"   - Ultralytics: {ultralytics.__version__}")
except ImportError:
    print("   ❌ Ultralytics not installed. Installing...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics"])
    import ultralytics
    print(f"   - Ultralytics: {ultralytics.__version__}")

import torch
print(f"   - PyTorch: {torch.__version__}")
print(f"   - CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   - GPU: {torch.cuda.get_device_name(0)}")
    device = '0'  # Gunakan GPU
else:
    print(f"   - Mode: CPU")
    device = 'cpu'

from ultralytics import YOLO

# ============================================
# 2. SETUP PATHS
# ============================================

project_root = Path.cwd().parent if Path.cwd().name in ['notebooks', 'notebook'] else Path.cwd()
processed_dir = project_root / "dataset" / "processed" / "yolo"
model_dir = project_root / "models" / "yolo"
result_dir = project_root / "results" / "yolo"

model_dir.mkdir(parents=True, exist_ok=True)
result_dir.mkdir(parents=True, exist_ok=True)

print(f"\n[Paths]")
print(f"   - Project: {project_root}")
print(f"   - Data: {processed_dir}")
print(f"   - Model: {model_dir}")
print(f"   - Results: {result_dir}")

# ============================================
# 3. CONFIGURATION
# ============================================

CONFIG = {
    'model_name': 'yolov8n-cls.pt',  # Nano version (paling ringan)
    'epochs': 30,                    # Sama dengan CNN
    'batch_size': 16,                # Sama dengan CNN
    'imgsz': 224,                    # Sama dengan CNN
    'learning_rate': 0.001,          # Sama dengan CNN
    'optimizer': 'AdamW',            # Optimizer standar
    'weight_decay': 0.0005,
    'dropout': 0.3,
    'patience': 10,                  # Early stopping
    'workers': 4,
    'device': device,                # Auto detect GPU/CPU
    'project': str(model_dir),
    'name': 'yolo_classification',
    'exist_ok': True,
    'verbose': True,
    'class_names': ['awake', 'drowsy']
}

print("\n[Configuration]")
for key, value in CONFIG.items():
    print(f"   - {key}: {value}")

# ============================================
# 4. LOAD PRETRAINED MODEL
# ============================================

print("\n[Loading Pretrained Model]")

model = YOLO(CONFIG['model_name'])
print(f"✅ Model loaded: {CONFIG['model_name']}")
print(f"   - Model type: Classification")

# ============================================
# 5. TRAINING
# ============================================

print("\n" + "="*60)
print("🎯 YOLOv8 TRAINING")
print("="*60)

start_time = time.time()

# Training parameters
train_params = {
    'data': str(processed_dir),  # Path ke direktori, BUKAN file .yaml
    'epochs': CONFIG['epochs'],
    'batch': CONFIG['batch_size'],
    'imgsz': CONFIG['imgsz'],
    'lr0': CONFIG['learning_rate'],
    'optimizer': CONFIG['optimizer'],
    'weight_decay': CONFIG['weight_decay'],
    'dropout': CONFIG['dropout'],
    'patience': CONFIG['patience'],
    'workers': CONFIG['workers'],
    'device': CONFIG['device'],
    'project': CONFIG['project'],
    'name': CONFIG['name'],
    'exist_ok': CONFIG['exist_ok'],
    'verbose': CONFIG['verbose'],
    'seed': 42,
}

print("\n[Training Parameters]")
for key, value in train_params.items():
    print(f"   - {key}: {value}")

print("\n[Training Progress]")
print("="*60)

results = model.train(**train_params)

training_time = time.time() - start_time
print(f"\n⏱️ Training time: {training_time/60:.2f} minutes")

# ============================================
# 6. SAVE BEST MODEL
# ============================================

print("\n[Saving Model]")

# Path ke best model
best_model_path = model_dir / CONFIG['name'] / 'weights' / 'best.pt'
if best_model_path.exists():
    # Copy ke folder models/yolo
    shutil.copy2(best_model_path, model_dir / 'best.pt')
    print(f"✅ Best model saved to: {model_dir / 'best.pt'}")
else:
    print(f"⚠️ Best model not found at: {best_model_path}")

# ============================================
# 7. LOAD BEST MODEL
# ============================================

print("\n[Loading Best Model]")

best_model = YOLO(model_dir / 'best.pt')

# ============================================
# 8. EVALUASI (DIPERBAIKI)
# ============================================

print("\n[Evaluating Model]")

# Evaluate on test set
test_results = best_model.val(
    data=str(processed_dir),  # Path ke direktori
    split='test',
    imgsz=CONFIG['imgsz'],
    batch=CONFIG['batch_size'],
    device=CONFIG['device'],
    verbose=False
)

# Extract metrics - PERBAIKI: loss tidak tersedia untuk klasifikasi
test_accuracy = test_results.top1  # Top-1 accuracy
test_accuracy_top5 = test_results.top5  # Top-5 accuracy

print(f"\n📊 Test Set Results:")
print(f"   - Top-1 Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"   - Top-5 Accuracy: {test_accuracy_top5:.4f} ({test_accuracy_top5*100:.2f}%)")

# ============================================
# 9. PREDICTIONS
# ============================================

print("\n[Generating Predictions]")

# Get test set predictions
test_path = processed_dir / 'test'
y_true = []
y_pred = []
y_pred_proba = []

# Process each class folder
for class_idx, class_name in enumerate(['awake', 'drowsy']):
    class_folder = test_path / class_name
    if class_folder.exists():
        image_files = list(class_folder.glob('*.jpg')) + list(class_folder.glob('*.png')) + list(class_folder.glob('*.jpeg'))
        
        for img_path in image_files:
            # Predict
            result = best_model(img_path, imgsz=CONFIG['imgsz'])
            
            # Get prediction
            probs = result[0].probs
            if probs is not None:
                pred_class = probs.top1
                pred_confidence = probs.top1conf.item()
                
                y_true.append(class_idx)
                y_pred.append(pred_class)
                y_pred_proba.append(pred_confidence if pred_class == 1 else 1 - pred_confidence)

print(f"   - Total predictions: {len(y_true)}")

# Convert to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_pred_proba = np.array(y_pred_proba)

# ============================================
# 10. CONFUSION MATRIX
# ============================================

print("\n[Confusion Matrix]")

cm = confusion_matrix(y_true, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CONFIG['class_names'],
            yticklabels=CONFIG['class_names'], ax=ax1)
ax1.set_title('Confusion Matrix - YOLOv8', fontsize=14, fontweight='bold')
ax1.set_ylabel('Actual')
ax1.set_xlabel('Predicted')

sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=CONFIG['class_names'],
            yticklabels=CONFIG['class_names'], ax=ax2)
ax2.set_title('Confusion Matrix (Normalized) - YOLOv8', fontsize=14, fontweight='bold')
ax2.set_ylabel('Actual')
ax2.set_xlabel('Predicted')

plt.tight_layout()
plt.savefig(result_dir / 'confusion_matrix.png', dpi=300)
plt.show()

# ============================================
# 11. CLASSIFICATION REPORT
# ============================================

print("\n[Classification Report]")

report = classification_report(y_true, y_pred, 
                               target_names=CONFIG['class_names'],
                               output_dict=True)

report_df = pd.DataFrame(report).transpose()
print("\n" + report_df.round(4).to_string())

# Extract metrics
precision = report['weighted avg']['precision']
recall = report['weighted avg']['recall']
f1_score = report['weighted avg']['f1-score']

print(f"\n📊 Weighted Average:")
print(f"   - Precision: {precision:.4f}")
print(f"   - Recall: {recall:.4f}")
print(f"   - F1-Score: {f1_score:.4f}")

# ============================================
# 12. ROC CURVE
# ============================================

print("\n[ROC Curve]")

fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='darkorange', lw=2, 
        label=f'YOLOv8 (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve - YOLOv8', fontsize=14, fontweight='bold')
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(result_dir / 'roc_curve.png', dpi=300)
plt.show()

# ============================================
# 13. TRAINING METRICS
# ============================================

print("\n[Training Metrics]")

# Extract training history from results
if hasattr(results, 'metrics'):
    print(f"   - Final metrics: {results.metrics}")

# ============================================
# 14. EXPORT RESULTS
# ============================================

print("\n[Exporting Results]")

timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

metrics = {
    'model': 'YOLOv8 Classification',
    'timestamp': timestamp,
    'training_time_minutes': training_time/60,
    'epochs': CONFIG['epochs'],
    'test_metrics': {
        'accuracy': float(test_accuracy),
        'top5_accuracy': float(test_accuracy_top5),
        'precision': float(precision),
        'recall': float(recall),
        'f1_score': float(f1_score),
        'auc_roc': float(roc_auc)
    },
    'config': CONFIG,
    'confusion_matrix': cm.tolist(),
    'classification_report': report
}

with open(result_dir / 'results.json', 'w') as f:
    json.dump(metrics, f, indent=4)

# Export classification report
report_df.to_csv(result_dir / 'classification_report.csv')

# Export confusion matrix
np.savetxt(result_dir / 'confusion_matrix.csv', cm, delimiter=',', fmt='%d')

print(f"✅ All results exported to: {result_dir}")

# ============================================
# 15. MODEL COMPARISON TABLE (DIPERBAIKI)
# ============================================

print("\n[Model Comparison]")

# Buat folder comparison
comparison_dir = project_root / 'results' / 'comparison'
comparison_dir.mkdir(parents=True, exist_ok=True)

# Load previous model results if available
comparison_data = {
    'Model': ['YOLOv8'],
    'Accuracy': [f"{test_accuracy*100:.2f}%"],
    'Precision': [f"{precision:.4f}"],
    'Recall': [f"{recall:.4f}"],
    'F1-Score': [f"{f1_score:.4f}"],
    'AUC-ROC': [f"{roc_auc:.4f}"],
    'Training Time': [f"{training_time/60:.1f} min"]
}

# Try to load CNN and MobileNetV2 results
for model_name in ['cnn', 'mobilenetv2']:
    result_file = project_root / 'results' / model_name / 'results.json'
    if result_file.exists():
        try:
            with open(result_file, 'r') as f:
                data = json.load(f)
            
            # Ambil data dengan aman (gunakan .get() untuk menghindari KeyError)
            test_metrics = data.get('test_metrics', {})
            accuracy = test_metrics.get('accuracy', 0)
            precision = test_metrics.get('precision', 0)
            recall = test_metrics.get('recall', 0)
            f1 = test_metrics.get('f1_score', 0)
            auc_roc = test_metrics.get('auc_roc', 0)
            
            # Cari training time di berbagai kemungkinan key
            training_time_val = data.get('training_time_minutes')
            if training_time_val is None:
                training_time_val = data.get('training_time', 0)
                if isinstance(training_time_val, (int, float)):
                    training_time_val = training_time_val / 60  # Konversi detik ke menit
                else:
                    training_time_val = 0
            
            comparison_data['Model'].append(model_name.upper())
            comparison_data['Accuracy'].append(f"{accuracy*100:.2f}%")
            comparison_data['Precision'].append(f"{precision:.4f}")
            comparison_data['Recall'].append(f"{recall:.4f}")
            comparison_data['F1-Score'].append(f"{f1:.4f}")
            comparison_data['AUC-ROC'].append(f"{auc_roc:.4f}")
            comparison_data['Training Time'].append(f"{training_time_val:.1f} min")
            
        except Exception as e:
            print(f"⚠️ Error loading {model_name}: {e}")
            # Tetap tambahkan baris dengan data kosong
            comparison_data['Model'].append(model_name.upper())
            comparison_data['Accuracy'].append("N/A")
            comparison_data['Precision'].append("N/A")
            comparison_data['Recall'].append("N/A")
            comparison_data['F1-Score'].append("N/A")
            comparison_data['AUC-ROC'].append("N/A")
            comparison_data['Training Time'].append("N/A")

# Create comparison DataFrame
comparison_df = pd.DataFrame(comparison_data)
print("\n📊 Perbandingan Model:")
print(comparison_df.to_string(index=False))

# Save comparison
comparison_df.to_csv(comparison_dir / 'model_comparison.csv', index=False)

# ============================================
# 16. RINGKASAN AKHIR
# ============================================

print("\n" + "="*60)
print("✅ YOLOv8 TRAINING COMPLETE!")
print("="*60)

print(f"\n📊 RINGKASAN HASIL:")
print(f"   - Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"   - Test Top-5 Accuracy: {test_accuracy_top5:.4f} ({test_accuracy_top5*100:.2f}%)")
print(f"   - Test Precision: {precision:.4f}")
print(f"   - Test Recall: {recall:.4f}")
print(f"   - Test F1-Score: {f1_score:.4f}")
print(f"   - AUC-ROC: {roc_auc:.4f}")
print(f"   - Training Time: {training_time/60:.2f} minutes")

print(f"\n📁 Hasil disimpan di: {result_dir}")
print(f"   - best.pt: {model_dir / 'best.pt'}")
print(f"   - results.json: {result_dir / 'results.json'}")
print(f"   - confusion_matrix.png: {result_dir / 'confusion_matrix.png'}")
print(f"   - roc_curve.png: {result_dir / 'roc_curve.png'}")
print(f"   - classification_report.csv: {result_dir / 'classification_report.csv'}")

print("\n" + "="*60)
print("🚀 LANJUT KE EVALUASI (06_Evaluation.ipynb)")
print("="*60)